In [0]:
from pyspark.sql import functions as F

In [0]:
def multi_col_apply(func):
    def wrapper(df, cols=None):
        if cols is None:
            cols = df.columns
        for c in cols:
            df = func(df, c)
        return df
    return wrapper

def apply_transformations(df, transformations):
    for func, kwargs in transformations:
        df = func(df, **kwargs)
    return df

In [0]:
@multi_col_apply
def trimming(df, col):
    return df.withColumn(col, F.trim(F.col(col)))

@multi_col_apply
def uppercase(df, col):
    return df.withColumn(col, F.upper(F.col(col)))

@multi_col_apply
def lowercase(df, col):
    return df.withColumn(col, F.lower(F.col(col)))

@multi_col_apply
def capitalize(df, col):
    return df.withColumn(col, F.initcap(F.col(col)))

def mobile_num_cleanup(df, col):
    new_col_name = col + '_cleaned'
    return (
        df
        .withColumn(new_col_name, F.regexp_replace(F.col(col), r"^\+\d{2}[\s\-\(\)]*|[\s\-\(\)]", ""))
        .withColumn(new_col_name, F.substring(new_col_name, -10,10))
    )
    

In [0]:
df_offline_customers = spark.table("retail_data.bronze.offline_customers")
df_online_customers = spark.table("retail_data.bronze.online_customers")

In [0]:
online_cust_transformations = [
    (mobile_num_cleanup, {"col": "mobile_number"}),
    (trimming, {}),
    (capitalize, {"cols": ["full_name", "city"]}),
    (lowercase, {"cols": ["email"]}),
    (uppercase, {"cols": ["customer_id", "state"]})
]

df_online_transformed = apply_transformations(df_online_customers, online_cust_transformations)

offline_cust_transformations = [
    (mobile_num_cleanup, {"col": "phone"}),
    (trimming, {}),
    (capitalize, {"cols": ["name", "city"]}),
    (lowercase, {"cols": ["email"]}),
    (uppercase, {"cols": ["cust_id", "phone"]})
]

df_offline_transformed = apply_transformations(df_offline_customers, offline_cust_transformations)